# Titanic Dataset Preprocessing

This notebook covers the full preprocessing pipeline:
1. Load the Dataset
2. Handle Missing Values
3. Encode Categorical Variables
4. Normalize Numerical Features
5. Split the Dataset


## Import Libraries

In [1]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

---
## Step 1 — Load the Dataset

Load `train.csv` using Pandas, then display the first 5 rows and the dataset shape.

In [2]:
# Load the Titanic dataset
df = pd.read_csv('train.csv')

# Display the first 5 rows to inspect column names and sample values
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Check the dataset dimensions: (rows, columns)
print("Shape of the dataset:", df.shape)

Shape of the dataset: (891, 12)


---
## Step 2 — Handle Missing Values

Three columns need attention:
| Column | Missing | Strategy |
|--------|---------|----------|
| `Age` | ~177 | Fill with **median** (robust to outliers) |
| `Cabin` | ~687 (77 %) | **Drop** — too many gaps |
| `Embarked` | 2 | Fill with **mode** (most frequent port) |

In [4]:
# Identify columns with missing values
print("Missing values per column (before):")
print(df.isnull().sum())

Missing values per column (before):
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [5]:
# Fill missing Age values with the median age
df['Age'].fillna(df['Age'].median(), inplace=True)

# Drop the Cabin column — over 77% of values are missing
df.drop(columns=['Cabin'], inplace=True)

# Fill the 2 missing Embarked values with the most frequent port ('S')
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Verify — should now show 0 for Age and Embarked, and Cabin is gone
print("Missing values per column (after):")
print(df.isnull().sum())

Missing values per column (after):
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


C:\Users\Omole Peter\AppData\Local\Temp\ipykernel_29772\4262217442.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\Omole Peter\AppData\Local\Temp\ipykernel_29772\4262217442.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

---
## Step 3 — Encode Categorical Variables

ML models require numerical input:
- **`Sex`** → binary: `male = 0`, `female = 1`
- **`Embarked`** → one-hot encoded into dummy columns (`Embarked_C`, `Embarked_Q`, `Embarked_S`)

In [6]:
# Binary encode the Sex column: male→0, female→1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

print("Sex column after encoding:")
print(df['Sex'].value_counts())

Sex column after encoding:
Sex
0    577
1    314
Name: count, dtype: int64


In [7]:
# One-hot encode the Embarked column
# drop_first=False keeps all three port columns for clarity
df = pd.get_dummies(df, columns=['Embarked'], drop_first=False)

print("Columns after encoding:")
print(df.columns.tolist())

Columns after encoding:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


---
## Step 4 — Normalize Numerical Features

**Min-Max Scaling** compresses `Age` and `Fare` to the **[0, 1]** range:

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

This prevents columns with large ranges (e.g. Fare up to 512) from dominating the model.

In [8]:
# Instantiate the Min-Max scaler
scaler = MinMaxScaler()

# Fit and transform Age and Fare columns in place
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

print("Age and Fare after normalization (min should be 0, max should be 1):")
df[['Age', 'Fare']].describe()

Age and Fare after normalization (min should be 0, max should be 1):


,Age,Fare
count,891.000000,891.000000
mean,0.363679,0.062858
std,0.163605,0.096995
min,0.000000,0.000000
25%,0.271174,0.015440
50%,0.346569,0.028213
75%,0.434531,0.060508
max,1.000000,1.000000


---
## Step 5 — Split the Dataset

Split into:
- **X** — feature columns (everything except `Survived` and non-informative text columns)
- **y** — target label (`Survived`)

Then apply an **80% training / 20% validation** split with `stratify=y` to preserve class balance.

In [9]:
# Drop the target and non-informative columns to form the feature matrix X
drop_cols = ['Survived', 'Name', 'Ticket', 'PassengerId']
X = df.drop(columns=drop_cols)

# Target variable
y = df['Survived']

print("Feature columns:", X.columns.tolist())
print("Target distribution:")
print(y.value_counts())

Feature columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
Target distribution:
Survived
0    549
1    342
Name: count, dtype: int64


In [10]:
# Split: 80% training, 20% validation
# stratify=y ensures the survivor ratio is preserved in both sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size  : {X_train.shape}")
print(f"Validation set size: {X_val.shape}")

Training set size  : (712, 9)
Validation set size: (179, 9)
